## Chat Bot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

os.environ["LANGCHAIN_TRACING"] = "true"


In [ ]:
## Create the datapoints
from langsmith import Client
client = Client()

## define the dataset - these are your test data
dataset_name = "Simple Kaushal Chatbots Evaluation"

# Only create the dataset once - re-running would raise a conflict error
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name)
    client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"question": "What is LangChain?"},
                "outputs": {"answer": "A framework for building LLM applications"},
            },
            {
                "inputs": {"question": "What is LangSmith?"},
                "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
            },
            {
                "inputs": {"question": "What is OpenAI?"},
                "outputs": {"answer": "A company that creates Large Language Models"},
            },
            {
                "inputs": {"question": "What is Google?"},
                "outputs": {"answer": "A technology company known for search"},
            },
            {
                "inputs": {"question": "What is Mistral?"},
                "outputs": {"answer": "A company that creates Large Language Models"},
            }
        ]
    )

## Define Metrics (LLM AS A Judge)

In [22]:
import anthropic
from langsmith import wrappers

# Initialize the Anthropic client
anthropic_client = wrappers.wrap_anthropic(
    anthropic.Anthropic()
)

eval_instructions = "You are an expert professor grading student answers."


# Your correctness evaluator (LLM as a judge)
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    # The target function failed for this example - nothing to grade
    if "response" not in outputs:
        return False

    user_content = f"""
Question:
{inputs['question']}

Reference answer:
{reference_outputs['answer']}

Predicted answer:
{outputs['response']}

Judge whether the predicted answer is semantically correct.
Reply with exactly CORRECT or INCORRECT.
"""

    result = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=20,
        system=eval_instructions,
        messages=[
            {"role": "user", "content": user_content}
        ]
    )

    return result.content[0].text.strip().upper() == "CORRECT"

In [23]:
## Concision - checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    if "response" not in outputs:
        return False
    return len(outputs["response"]) < 2 * len(reference_outputs["answer"])


## Run Evaluations

In [24]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

# Your application function
def my_app(question: str, instructions: str = default_instructions) -> str:
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=instructions,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.content[0].text

In [25]:
### Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(inputs["question"])
    }

In [26]:

## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="claude-sonnet-4-6-chatbot"
)

View the evaluation results for experiment: 'claude-sonnet-4-6-chatbot-959087d3' at:
https://smith.langchain.com/o/62eb9a75-1b2a-45f3-b5f3-3c626cd91ac1/datasets/e8c6db1d-bb04-418c-ab97-748dcb243ce9/compare?selectedSessions=63972bab-591b-4ba1-bca3-63a565b8d23f




5it [00:14,  2.99s/it]
